# Week 03: PHASE 1 — 2D Kinematics — The Language of Motion

*Physics I (PHY101) . 3 Hours . Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this session you will be able to:

1. **Decompose** a velocity vector into horizontal and vertical components
2. **Derive** and apply the equations of projectile motion (range, maximum height, time of flight)
3. **Predict** where a projectile lands given initial speed and launch angle
4. **Compare** ideal (vacuum) trajectories with air-resistance trajectories
5. **Optimize** launch angle for maximum range and for hitting a specific target

## Core Mastery Connection

**PHASE 1 — "The Language of Motion":** Decompose 2D motion into independent axes — predict each separately, combine for the trajectory. Projectile motion is the capstone of Phase 1: you use vector decomposition (Week 1) to split the problem into two independent 1D kinematic problems (Week 2), then combine the results to predict the full parabolic trajectory. This is exactly the core mastery workflow in action.

---
## 1. Setup

Run the cell below to import everything we need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

# Nicer default plots
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 13,
    'axes.grid': True,
    'grid.alpha': 0.3
})

g = 9.81  # m/s^2
print("All imports ready.")

---
## 2. Theory: Projectile Motion

### 2.1 What is a Projectile?

A **projectile** is any object that, once launched, moves under the influence of gravity alone (we ignore air resistance for now).

| Quantity | Symbol | Formula |
|---|---|---|
| Initial speed | $v_0$ | given |
| Launch angle | $\theta$ | given |
| Horizontal component | $v_{0x}$ | $v_0 \cos\theta$ |
| Vertical component | $v_{0y}$ | $v_0 \sin\theta$ |
| Horizontal position | $x(t)$ | $v_{0x}\, t$ |
| Vertical position | $y(t)$ | $v_{0y}\, t - \tfrac{1}{2}g\,t^2$ |
| Time of flight | $T$ | $\dfrac{2\,v_0 \sin\theta}{g}$ |
| Maximum height | $H$ | $\dfrac{v_0^2 \sin^2\theta}{2g}$ |
| Range | $R$ | $\dfrac{v_0^2 \sin 2\theta}{g}$ |

### 2.2 Key Insight — Independence of Motion

Horizontal and vertical motions are **independent**:
- Horizontally: constant velocity (no acceleration)
- Vertically: constant acceleration $g$ downward

> **Analogy:** Imagine two balls released at the same instant — one dropped from a cliff, the other thrown horizontally. Both hit the ground at the **same time** because the vertical motion is identical.

### 2.3 Trajectory Equation (Eliminating Time)

Solving $x = v_{0x} t$ for $t$ and substituting into $y(t)$:

$$y = x\tan\theta - \frac{g\,x^2}{2\,v_0^2\cos^2\theta}$$

This is a **parabola** — the hallmark shape of projectile motion.

---
## 3. Interactive Demo 1 — Animated Projectile Trajectory

Watch a projectile fly through the air. Adjust the **launch angle** and **initial speed** with the sliders, then click **Launch!**

In [ ]:
def projectile_trajectory(v0, theta_deg):
    """Return x, y arrays and key quantities for ideal projectile."""
    theta = np.radians(theta_deg)
    T = 2 * v0 * np.sin(theta) / g
    t = np.linspace(0, T, 300)
    x = v0 * np.cos(theta) * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    R = v0**2 * np.sin(2 * theta) / g
    H = v0**2 * np.sin(theta)**2 / (2 * g)
    return t, x, y, T, R, H


def animate_projectile(v0=25, theta_deg=45):
    """Create an animated projectile trajectory."""
    t, x, y, T, R, H = projectile_trajectory(v0, theta_deg)

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.set_xlim(-2, max(R * 1.15, 10))
    ax.set_ylim(-1, max(H * 1.4, 5))
    ax.set_xlabel('Horizontal Distance (m)')
    ax.set_ylabel('Height (m)')
    ax.set_title(f'Projectile Motion  —  $v_0$={v0} m/s,  $\\theta$={theta_deg}°')
    ax.set_aspect('equal')

    # Ground
    ax.axhline(0, color='saddlebrown', linewidth=2)
    ax.fill_between(ax.get_xlim(), -1, 0, color='saddlebrown', alpha=0.15)

    trail, = ax.plot([], [], 'b-', linewidth=1.5, alpha=0.5, label='Trajectory')
    ball, = ax.plot([], [], 'ro', markersize=10)
    info_text = ax.text(0.02, 0.95, '', transform=ax.transAxes,
                        fontsize=11, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Mark key points
    ax.plot(R / 2, H, 'g^', markersize=12, label=f'Max height = {H:.1f} m')
    ax.plot(R, 0, 'kx', markersize=14, markeredgewidth=3, label=f'Range = {R:.1f} m')
    ax.legend(loc='upper right')

    n_frames = len(t)
    skip = max(1, n_frames // 120)  # ~120 frames for smooth animation

    def init():
        trail.set_data([], [])
        ball.set_data([], [])
        info_text.set_text('')
        return trail, ball, info_text

    def update(frame):
        idx = frame * skip
        if idx >= n_frames:
            idx = n_frames - 1
        trail.set_data(x[:idx+1], y[:idx+1])
        ball.set_data([x[idx]], [y[idx]])
        info_text.set_text(f't = {t[idx]:.2f} s\nx = {x[idx]:.1f} m\ny = {y[idx]:.1f} m')
        return trail, ball, info_text

    total_frames = n_frames // skip + 1
    anim = FuncAnimation(fig, update, init_func=init,
                         frames=total_frames, interval=30, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Interactive widget
v0_slider = widgets.IntSlider(value=30, min=5, max=60, step=1,
                              description='v₀ (m/s):')
angle_slider = widgets.IntSlider(value=45, min=5, max=85, step=1,
                                 description='θ (deg):')
launch_btn = widgets.Button(description='🚀 Launch!',
                            button_style='success', layout=widgets.Layout(width='150px'))
output = widgets.Output()

def on_launch(b):
    with output:
        clear_output(wait=True)
        display(animate_projectile(v0_slider.value, angle_slider.value))

launch_btn.on_click(on_launch)
display(widgets.VBox([widgets.HBox([v0_slider, angle_slider, launch_btn]), output]))

# Auto-launch default
on_launch(None)

---
## 4. Interactive Demo 2 — Range vs Launch Angle Optimizer

How does range $R$ depend on launch angle? Move the slider and watch the highlighted point on the curve. The **optimal angle for maximum range** (on level ground) is always **45°**.

In [ ]:
def range_vs_angle(v0=30):
    """Interactive range-vs-angle plot."""
    angles = np.linspace(0, 90, 500)
    ranges = v0**2 * np.sin(2 * np.radians(angles)) / g

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(angles, ranges, 'b-', linewidth=2)
    ax.set_xlabel('Launch Angle (°)')
    ax.set_ylabel('Range (m)')
    ax.set_title(f'Range vs Launch Angle  (v₀ = {v0} m/s)')

    # Highlight 45°
    R_max = v0**2 / g
    ax.axvline(45, color='r', linestyle='--', alpha=0.5)
    ax.plot(45, R_max, 'r*', markersize=18, label=f'Max range = {R_max:.1f} m at 45°')

    # Show complementary angles give same range
    for a in [30, 60]:
        R_a = v0**2 * np.sin(2 * np.radians(a)) / g
        ax.plot(a, R_a, 'go', markersize=10)
        ax.annotate(f'{a}°: R={R_a:.1f} m', (a, R_a),
                    textcoords='offset points', xytext=(10, 10), fontsize=10)

    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()


@widgets.interact(v0=widgets.IntSlider(value=30, min=5, max=80, step=5,
                                       description='v₀ (m/s):'))
def _range_plot(v0):
    range_vs_angle(v0)

### 4.1 Complementary Angles

Notice that **complementary angles** ($\theta$ and $90°-\theta$) give the **same range**:

$$\sin(2\theta) = \sin(2(90°-\theta)) = \sin(180°-2\theta) = \sin(2\theta)$$

For example, 30° and 60° produce identical ranges but very different trajectories (30° is flat, 60° is high).

---
## 5. Interactive Demo 3 — Vacuum vs Air Resistance

Real projectiles experience **air drag**. We model the drag force as:

$$\vec{F}_{\text{drag}} = -\tfrac{1}{2}\,C_d\,\rho\,A\,|\vec{v}|\,\vec{v}$$

where $C_d$ is the drag coefficient, $\rho$ is air density, and $A$ is cross-sectional area.

For simplicity we combine these into a single **drag parameter** $b = \tfrac{1}{2}C_d \rho A / m$.

In [ ]:
def simulate_with_drag(v0, theta_deg, b=0.0, dt=0.005):
    """Numerically integrate projectile with quadratic drag.
    b = drag parameter (0 = vacuum).
    Returns arrays x, y.
    """
    theta = np.radians(theta_deg)
    vx, vy = v0 * np.cos(theta), v0 * np.sin(theta)
    x_pos, y_pos = [0.0], [0.0]

    while y_pos[-1] >= 0 or len(x_pos) < 3:
        speed = np.sqrt(vx**2 + vy**2)
        ax_drag = -b * speed * vx
        ay_drag = -b * speed * vy
        vx += ax_drag * dt
        vy += (-g + ay_drag) * dt
        x_pos.append(x_pos[-1] + vx * dt)
        y_pos.append(y_pos[-1] + vy * dt)
        if len(x_pos) > 100_000:
            break

    return np.array(x_pos), np.array(y_pos)


def compare_drag(v0=30, theta_deg=45, b=0.02):
    """Side-by-side comparison: ideal vs drag."""
    # Ideal
    _, x_ideal, y_ideal, _, R_ideal, H_ideal = projectile_trajectory(v0, theta_deg)
    # With drag
    x_drag, y_drag = simulate_with_drag(v0, theta_deg, b)

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(x_ideal, y_ideal, 'b-', linewidth=2.5, label='Ideal (vacuum)')
    ax.plot(x_drag, y_drag, 'r--', linewidth=2.5, label=f'With drag (b={b})')
    ax.axhline(0, color='saddlebrown', linewidth=2)
    ax.fill_between([ax.get_xlim()[0], max(x_ideal.max(), x_drag.max()) * 1.1],
                    -2, 0, color='saddlebrown', alpha=0.12)
    ax.set_xlabel('Horizontal Distance (m)')
    ax.set_ylabel('Height (m)')
    ax.set_title(f'Ideal vs Air Resistance  —  v₀={v0} m/s, θ={theta_deg}°')
    ax.set_ylim(bottom=-1)
    ax.legend(fontsize=12)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

    # Find drag range (first y<0 after launch)
    idx_land = np.argmax(y_drag[2:] < 0) + 2
    R_drag = x_drag[idx_land] if idx_land < len(x_drag) else x_drag[-1]
    print(f"  Ideal range : {R_ideal:.1f} m")
    print(f"  Drag range  : {R_drag:.1f} m  ({(1 - R_drag/R_ideal)*100:.1f}% reduction)")


@widgets.interact(
    v0=widgets.IntSlider(value=30, min=5, max=80, step=5, description='v₀ (m/s):'),
    theta_deg=widgets.IntSlider(value=45, min=5, max=85, step=1, description='θ (°):'),
    b=widgets.FloatSlider(value=0.02, min=0.0, max=0.15, step=0.005,
                          description='Drag b:', readout_format='.3f')
)
def _drag_compare(v0, theta_deg, b):
    compare_drag(v0, theta_deg, b)

---
## 6. Interactive Demo 4 — Target Hitting Game 🎯

A target sits at a specific distance and height. Your job: choose the **launch angle** to hit it!

Adjust the angle slider and press **Fire!** The projectile must pass within 1 m of the target centre.

In [ ]:
np.random.seed(42)

# Game state
V0_GAME = 35  # fixed speed
TARGET_X = np.random.uniform(40, 90)
TARGET_Y = np.random.uniform(0, 15)
HIT_RADIUS = 1.5  # metres

angle_game = widgets.FloatSlider(value=45, min=5, max=85, step=0.5,
                                 description='θ (°):', readout_format='.1f')
fire_btn = widgets.Button(description='🔥 Fire!', button_style='danger',
                          layout=widgets.Layout(width='120px'))
new_btn = widgets.Button(description='🎲 New Target', button_style='info',
                         layout=widgets.Layout(width='150px'))
game_out = widgets.Output()


def draw_game(theta_deg=None):
    global TARGET_X, TARGET_Y
    fig, ax = plt.subplots(figsize=(12, 6))

    # Ground
    ax.axhline(0, color='saddlebrown', linewidth=2)
    x_max = max(TARGET_X * 1.3, 60)
    ax.set_xlim(-3, x_max)
    ax.set_ylim(-3, max(40, TARGET_Y + 20))

    # Target
    target_circle = plt.Circle((TARGET_X, TARGET_Y), HIT_RADIUS,
                               color='red', alpha=0.4, linewidth=2)
    ax.add_patch(target_circle)
    ax.plot(TARGET_X, TARGET_Y, 'r+', markersize=20, markeredgewidth=3)
    ax.annotate(f'Target ({TARGET_X:.0f}, {TARGET_Y:.0f})',
                (TARGET_X, TARGET_Y), textcoords='offset points',
                xytext=(10, 10), fontsize=11, color='red')

    hit = False
    if theta_deg is not None:
        _, x, y, _, _, _ = projectile_trajectory(V0_GAME, theta_deg)
        mask = y >= 0
        ax.plot(x[mask], y[mask], 'b-', linewidth=2, label=f'θ = {theta_deg:.1f}°')

        # Check hit
        dist = np.sqrt((x - TARGET_X)**2 + (y - TARGET_Y)**2)
        min_dist = dist.min()
        if min_dist <= HIT_RADIUS:
            hit = True
            ax.set_title('🎯  HIT!  Great shot!', fontsize=18, color='green', fontweight='bold')
        else:
            ax.set_title(f'MISS — closest approach: {min_dist:.1f} m', fontsize=15, color='gray')
        ax.legend(fontsize=12)
    else:
        ax.set_title(f'Target Hitting Game  —  v₀ = {V0_GAME} m/s  (adjust angle & fire!)',
                     fontsize=14)

    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    return hit


def on_fire(b):
    with game_out:
        clear_output(wait=True)
        draw_game(angle_game.value)


def on_new(b):
    global TARGET_X, TARGET_Y
    TARGET_X = np.random.uniform(40, 90)
    TARGET_Y = np.random.uniform(0, 15)
    with game_out:
        clear_output(wait=True)
        draw_game()


fire_btn.on_click(on_fire)
new_btn.on_click(on_new)
display(widgets.VBox([widgets.HBox([angle_game, fire_btn, new_btn]), game_out]))
on_new(None)  # show initial target

---
## 7. Interactive Demo 5 — Multiple Trajectories at Different Angles

This demo overlays trajectories for a range of launch angles so you can visually compare their shapes, heights, and ranges.

In [ ]:
def multi_trajectory(v0=30):
    """Plot trajectories for multiple angles on the same axes."""
    fig, ax = plt.subplots(figsize=(12, 6))
    angles = [15, 30, 45, 60, 75]
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(angles)))

    for angle, color in zip(angles, cmap):
        _, x, y, _, R, H = projectile_trajectory(v0, angle)
        mask = y >= 0
        ax.plot(x[mask], y[mask], color=color, linewidth=2,
                label=f'{angle}° (R={R:.0f} m, H={H:.0f} m)')

    ax.axhline(0, color='saddlebrown', linewidth=2)
    ax.set_xlabel('Horizontal Distance (m)')
    ax.set_ylabel('Height (m)')
    ax.set_title(f'Trajectories at Different Angles  (v₀ = {v0} m/s)')
    ax.legend(loc='upper right')
    ax.set_ylim(bottom=-1)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()


@widgets.interact(v0=widgets.IntSlider(value=30, min=10, max=60, step=5,
                                       description='v₀ (m/s):'))
def _multi(v0):
    multi_trajectory(v0)

---
## 8. Worked Examples

### Example 1: Football Kick

A football is kicked with $v_0 = 20$ m/s at $\theta = 37°$. Find:
- (a) Time of flight
- (b) Maximum height
- (c) Range

In [ ]:
v0_ex1 = 20  # m/s
theta_ex1 = 37  # degrees
theta_rad = np.radians(theta_ex1)

T_ex1 = 2 * v0_ex1 * np.sin(theta_rad) / g
H_ex1 = v0_ex1**2 * np.sin(theta_rad)**2 / (2 * g)
R_ex1 = v0_ex1**2 * np.sin(2 * theta_rad) / g

print(f"Football kick: v₀ = {v0_ex1} m/s, θ = {theta_ex1}°")
print(f"  (a) Time of flight : T = {T_ex1:.2f} s")
print(f"  (b) Maximum height : H = {H_ex1:.2f} m")
print(f"  (c) Range          : R = {R_ex1:.2f} m")

### Example 2: Cliff Launch

A ball is thrown horizontally from a cliff 45 m high with $v_0 = 15$ m/s.
How far from the base of the cliff does it land?

In [ ]:
h_cliff = 45  # m
v0_ex2 = 15  # m/s (horizontal)

# Time to fall h: h = 0.5 * g * t^2
t_fall = np.sqrt(2 * h_cliff / g)
x_land = v0_ex2 * t_fall

print(f"Cliff height: {h_cliff} m, horizontal speed: {v0_ex2} m/s")
print(f"  Time to hit ground: {t_fall:.2f} s")
print(f"  Horizontal distance: {x_land:.2f} m")

# Visualize
t_arr = np.linspace(0, t_fall, 200)
x_arr = v0_ex2 * t_arr
y_arr = h_cliff - 0.5 * g * t_arr**2

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(x_arr, y_arr, 'b-', linewidth=2)
ax.plot(x_arr[-1], y_arr[-1], 'rx', markersize=15, markeredgewidth=3)
# Cliff face
ax.plot([0, 0], [0, h_cliff], 'k-', linewidth=4)
ax.axhline(0, color='saddlebrown', linewidth=2)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Horizontal Launch from a Cliff')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## Problem Set

**Core Mastery Workflow — For each problem: Draw the diagram -> Identify the principle -> Write the equation -> Predict -> Verify.**

**Instructions:** Solve each problem in the code cell provided. Show your work using Python calculations. Use `numpy` functions where appropriate. Problems are graded by difficulty:
- **L1 (Basic):** Single-concept, straightforward calculation
- **L2 (Intermediate):** Multi-step, combines two or more concepts
- **L3 (Challenge):** Multi-concept integration, deeper analysis required

### L1 -- P1: Basic Projectile Range

A soccer ball is kicked from ground level with an initial speed of 22.0 m/s at an angle of 40° above the horizontal. Find (a) the time of flight, (b) the maximum height, and (c) the range. Ignore air resistance.

<details><summary>Answer</summary>(a) T = 2v0 sinθ/g = 2(22)sin40°/9.81 = 2.88 s. (b) H = v0² sin²θ/(2g) = 484(0.413)/19.62 = 10.19 m. (c) R = v0² sin2θ/g = 484 sin80°/9.81 = 48.6 m.</details>

In [ ]:
# ✏️ [P1] Your solution here


### L1 -- P2: Horizontal Launch

A marble rolls off the edge of a table 1.20 m above the floor with a horizontal speed of 2.50 m/s. (a) How long does it take to hit the floor? (b) How far from the base of the table does it land?

<details><summary>Answer</summary>(a) t = √(2h/g) = √(2.40/9.81) = 0.495 s. (b) x = v0 × t = 2.50 × 0.495 = 1.24 m.</details>

In [ ]:
# ✏️ [P2] Your solution here


### L1 -- P3: Projectile Components

A ball is launched at 18.0 m/s at 55° above the horizontal. What are the horizontal and vertical components of its velocity (a) at launch, (b) at the highest point, and (c) just before it returns to launch height?

<details><summary>Answer</summary>(a) vx = 18cos55° = 10.3 m/s, vy = 18sin55° = 14.7 m/s. (b) vx = 10.3 m/s, vy = 0. (c) vx = 10.3 m/s, vy = −14.7 m/s (symmetric).</details>

In [ ]:
# ✏️ [P3] Your solution here


### L1 -- P4: Time to Reach Height

A ball is thrown vertically upward at 25.0 m/s from the ground. At what two times does it pass through a height of 20.0 m?

<details><summary>Answer</summary>20 = 25t − 4.905t² → 4.905t² − 25t + 20 = 0 → t = (25 ± √(625−392.4))/9.81 = (25 ± 15.26)/9.81. t1 = 0.99 s (going up), t2 = 4.10 s (coming down).</details>

In [ ]:
# ✏️ [P4] Your solution here


### L2 -- P5: Cliff Launch at an Angle

A rescue package is launched from a cliff 60.0 m above the water at 20.0 m/s at 35° above the horizontal. (a) How long until it hits the water? (b) How far from the base of the cliff does it land? (c) What is its speed just before impact?

<details><summary>Answer</summary>(a) y = y0 + v0y t − ½gt²: 0 = 60 + 20sin35° t − 4.905t² → 4.905t² − 11.47t − 60 = 0 → t = (11.47 + √(131.6 + 1177.2))/9.81 = 4.86 s. (b) x = 20cos35° × 4.86 = 79.6 m. (c) vx = 16.38 m/s, vy = 11.47 − 9.81(4.86) = −36.2 m/s; v = √(268.3 + 1310.4) = 39.7 m/s.</details>

In [ ]:
# ✏️ [P5] Your solution here


### L2 -- P6: Complementary Angles

A cannon fires a shell at 50.0 m/s. (a) Find the two launch angles that produce a range of 200 m on level ground. (b) For each angle, find the maximum height and time of flight. (c) Which trajectory would be preferred for hitting a target behind a 15 m wall located 150 m away?

<details><summary>Answer</summary>(a) R = v0² sin2θ/g → sin2θ = Rg/v0² = 200(9.81)/2500 = 0.7848 → 2θ = 51.7° or 128.3° → θ = 25.9° or 64.1°. (b) θ=25.9°: H = 2500 sin²25.9°/19.62 = 24.3 m, T = 2(50)sin25.9°/9.81 = 4.45 s. θ=64.1°: H = 2500 sin²64.1°/19.62 = 103.1 m, T = 9.19 s. (c) At x=150 m with θ=25.9°: t=150/(50cos25.9°)=3.34 s, y=50sin25.9(3.34)−4.905(11.16)=72.9−54.7=18.2 m > 15 m; θ=64.1° also clears. Either works, but 25.9° has less flight time.</details>

In [ ]:
# ✏️ [P6] Your solution here


### L2 -- P7: Relative Velocity and River Crossing

A boat can travel at 4.0 m/s in still water. The river is 80.0 m wide and flows east at 3.0 m/s. (a) If the boat aims straight north, where does it end up relative to its starting point? (b) At what angle upstream should the boat aim to travel directly north? (c) How long does the crossing take in each case?

<details><summary>Answer</summary>(a) t = 80/4.0 = 20.0 s; drifts east 3.0 × 20 = 60 m; ends up 80 m north and 60 m east, total displacement = 100 m. (b) sinθ = 3.0/4.0 → θ = 48.6° upstream (west of north). (c) Case (a): 20.0 s. Case (b): v_north = √(16−9) = 2.65 m/s; t = 80/2.65 = 30.2 s.</details>

In [ ]:
# ✏️ [P7] Your solution here


### L2 -- P8: Projectile Meeting Point

Two balls are launched simultaneously from the same point on level ground. Ball A is launched at 30.0 m/s at 60°, and Ball B at 30.0 m/s at 30°. (a) Show that they have the same range. (b) At what time after launch are they at the same height? (c) What is that height?

<details><summary>Answer</summary>(a) sin(2×60°) = sin120° = sin(2×30°) = sin60° = 0.866 → same R = 79.5 m. (b) Heights: yA = 30sin60 t − 4.905t² = 25.98t − 4.905t²; yB = 15t − 4.905t². Set equal: 25.98t = 15t → only equal at t = 0 (they start together). Actually they are at same height at t=0 and when both return to ground. For intermediate crossing: 25.98t − 4.905t² = 15t − 4.905t² → 10.98t = 0 → t=0. They never cross at same height except at launch and landing. (c) At landing, h = 0 m.</details>

In [ ]:
# ✏️ [P8] Your solution here


### L3 -- P9: Projectile on an Incline

A ball is launched from the base of a hill that makes an angle α = 20° with the horizontal. The launch speed is 25.0 m/s and the launch angle is 50° above the horizontal (not above the incline). (a) Find the time at which the ball hits the incline. (Hint: the ball hits when y/x = tanα.) (b) Find the distance along the incline from the launch point to the impact point.

<details><summary>Answer</summary>(a) x = 25cos50° t = 16.07t; y = 25sin50° t − 4.905t² = 19.15t − 4.905t². Condition: y/x = tan20° = 0.364. (19.15t − 4.905t²)/(16.07t) = 0.364 → 19.15 − 4.905t = 5.851 → t = 13.30/4.905 = 2.71 s. (b) x = 16.07(2.71) = 43.6 m; y = 19.15(2.71) − 4.905(7.34) = 51.9 − 36.0 = 15.9 m. Distance along incline = x/cos20° = 43.6/0.940 = 46.4 m.</details>

In [ ]:
# ✏️ [P9] Your solution here


### L3 -- P10: Drone Delivery Problem

A delivery drone flies horizontally at a height of 40.0 m and a speed of 12.0 m/s toward a delivery zone. It must release a package so that it lands in a 2.0 m × 2.0 m target centered 50.0 m horizontally ahead of the drone's current position. (a) At what horizontal distance before the target should the drone release the package? (b) What is the acceptable range of release distances (to land within the 2 m target width)? (c) What is the package's speed and angle of impact?

<details><summary>Answer</summary>(a) Fall time: t = √(2×40/9.81) = 2.856 s. Horizontal distance during fall: d = 12.0 × 2.856 = 34.3 m. Release when target is 34.3 m ahead (i.e., 50.0 − 34.3 = 15.7 m from current position). (b) For ±1 m landing tolerance: t_near = d_near/12; h = 4.905t². At 33.3 m: t = 2.775 s, h = 37.8 m (not 40 m). Solve properly: targets at x = 49–51 m from release. t = √(80/9.81)=2.856 s; x = 12t = 34.27 m. For x = 33.27 and 35.27 m: Δt = ±1/12 = 0.083 s. Acceptable release window = ±1.0 m around the ideal point. (c) vx = 12.0 m/s, vy = 9.81 × 2.856 = 28.0 m/s; v = √(144+784) = 30.5 m/s at angle arctan(28.0/12.0) = 66.8° below horizontal.</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## 9. Bridge to Next Week

This week we described **how** objects move (kinematics). Next week we ask **why** they move — that is the domain of **Newton's Laws**.

Key connections:
- The parabolic trajectory arises because the only force is gravity ($\vec{F} = m\vec{g}$)
- Air drag adds a velocity-dependent force, which Newton's second law will let us handle systematically
- Free body diagrams will become our main tool for setting up equations of motion

**Next week:** *Newton's Laws (I) — Forces & Free Body Diagrams*